In [1]:
import json

input_path  = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs_chat.jsonl"
output_path = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs_final.jsonl"

refusal_phrase = "don't have enough information"

# Java domain keywords — if none present, record is off-topic
java_keywords = [
    "java","class","object","method","interface","array","loop",
    "inheritance","polymorphism","collection","list","map","set",
    "exception","thread","stream","lambda","string","int","void",
    "extends","implements","abstract","static","public","private",
    "constructor","package","import","iterator","generics","queue",
    "stack","tree","hash","sort","compile","runtime","jvm","byte"
]

removed_offtopic  = 0
removed_copypaste = 0
removed_short     = 0
removed_no_ctx    = 0
kept              = 0

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue
        item       = json.loads(line)
        user       = item["messages"][1]["content"]
        ans        = item["messages"][2]["content"]
        is_refusal = refusal_phrase in ans.lower()

        # ✅ Always keep intentional refusals
        if is_refusal:
            fout.write(json.dumps(item) + "\n")
            kept += 1
            continue

        combined = (user + " " + ans).lower()
        ctx      = user.split("Question:")[0].replace("Context:","").strip()

        # ❌ Remove off-topic (no Java keywords)
        if not any(kw in combined for kw in java_keywords):
            removed_offtopic += 1
            continue

        # ❌ Remove answerable with no context
        if len(ctx) < 10:
            removed_no_ctx += 1
            continue

        # ❌ Remove very short answers
        if len(ans.split()) <= 3:
            removed_short += 1
            continue

        # ❌ Remove copy-paste answers
        if len(ans) > 20 and ans.lower() in ctx.lower():
            removed_copypaste += 1
            continue

        fout.write(json.dumps(item) + "\n")
        kept += 1

print(f"Removed off-topic    : {removed_offtopic}")
print(f"Removed copy-paste   : {removed_copypaste}")
print(f"Removed no-context   : {removed_no_ctx}")
print(f"Removed short answers: {removed_short}")
print(f"Final clean records  : {kept}")
print(f"Saved to             : qa_pairs_final.jsonl")

Removed off-topic    : 1572
Removed copy-paste   : 2727
Removed no-context   : 24
Removed short answers: 100
Final clean records  : 7957
Saved to             : qa_pairs_final.jsonl


In [2]:
import json

input_path  = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs_final.jsonl"
output_path = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs_100.jsonl"

refusal_phrase = "don't have enough information"

java_keywords = [
    "java","class","object","method","interface","array","loop",
    "inheritance","polymorphism","collection","list","map","set",
    "exception","thread","stream","lambda","string","int","void",
    "extends","implements","abstract","static","public","private",
    "constructor","package","import","iterator","generics","queue",
    "stack","tree","hash","sort","compile","runtime","jvm","byte",
    "boolean","variable","parameter","return","recursive","algorithm",
    "data structure","linked","binary","null","instantiate","override"
]

blocklist = [
    "typography","leading","printing blocks","font size",
    "move disk","stack 0","stack 1","stack 2",
    "creative commons","license","pdf version","epub",
    "chapter exercise","quiz answer","textbook"
]

kept              = 0
removed_offtopic  = 0
removed_blocklist = 0

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue
        item       = json.loads(line)
        user       = item["messages"][1]["content"]
        ans        = item["messages"][2]["content"]
        is_refusal = refusal_phrase in ans.lower()

        # ✅ Always keep intentional refusals
        if is_refusal:
            fout.write(json.dumps(item) + "\n")
            kept += 1
            continue

        combined = (user + " " + ans).lower()

        # ❌ Remove blocklist content
        if any(bl in combined for bl in blocklist):
            removed_blocklist += 1
            continue

        # ❌ Remove off-topic (no Java keywords)
        if not any(kw in combined for kw in java_keywords):
            removed_offtopic += 1
            continue

        fout.write(json.dumps(item) + "\n")
        kept += 1

print(f"Removed blocklist    : {removed_blocklist}")
print(f"Removed off-topic    : {removed_offtopic}")
print(f"Final clean records  : {kept}")
print(f"Saved to             : qa_pairs_100.jsonl")

Removed blocklist    : 36
Removed off-topic    : 0
Final clean records  : 7921
Saved to             : qa_pairs_100.jsonl
